# RBAC Toy Model: Users, Roles, Permissions, and Sessions


## Teaching goal

This is a toy model of Role-Based Access Control (RBAC). It shows how permissions are assigned to
roles, users are assigned to roles, and sessions activate only the roles needed for a task. Real EHRs,
identity platforms, and operating systems provide RBAC features; administrators usually design the
roles, permissions, constraints, and review process.


In [ ]:
from dataclasses import dataclass
from pprint import pprint


@dataclass(frozen=True)
class Request:
    user: str
    operation: str
    resource: str
    context: dict


users = {
    "dr_rossi": {"name": "Dr. Rossi", "department": "cardiology"},
    "nurse_amina": {"name": "Nurse Amina", "department": "ward-a"},
    "lab_tech": {"name": "Lab Technician", "department": "laboratory"},
    "billing_clerk": {"name": "Billing Clerk", "department": "billing"},
    "privacy_auditor": {"name": "Privacy Auditor", "department": "compliance"},
}

resources = {
    "ehr_note_42": {"type": "ehr_note", "patient": "patient-42", "department": "cardiology"},
    "lab_result_42": {"type": "lab_result", "patient": "patient-42", "department": "laboratory"},
    "billing_record_42": {"type": "billing_record", "patient": "patient-42", "department": "billing"},
    "audit_log": {"type": "audit_log", "patient": None, "department": "compliance"},
}

requests = [
    Request("dr_rossi", "read", "ehr_note_42", {"assigned_patient": True, "emergency": False}),
    Request("nurse_amina", "write", "ehr_note_42", {"assigned_patient": True, "emergency": False}),
    Request("lab_tech", "write", "lab_result_42", {"assigned_patient": False, "emergency": False}),
    Request("billing_clerk", "read", "ehr_note_42", {"assigned_patient": False, "emergency": False}),
    Request("privacy_auditor", "read", "audit_log", {"assigned_patient": False, "emergency": False}),
]


In [ ]:
# Users may be assigned multiple roles.
user_roles = {
    "dr_rossi": {"attending_physician", "researcher"},
    "nurse_amina": {"ward_nurse"},
    "lab_tech": {"lab_technician"},
    "billing_clerk": {"billing_staff"},
    "privacy_auditor": {"auditor"},
}

# Roles carry permissions. A permission is represented as (operation, resource_type).
role_permissions = {
    "attending_physician": {("read", "ehr_note"), ("write", "ehr_note"), ("read", "lab_result")},
    "ward_nurse": {("read", "ehr_note"), ("write", "ehr_note"), ("read", "lab_result")},
    "lab_technician": {("read", "lab_result"), ("write", "lab_result")},
    "billing_staff": {("read", "billing_record"), ("write", "billing_record")},
    "auditor": {("read", "audit_log")},
    "researcher": {("read", "deidentified_dataset")},
}

# A session activates a subset of assigned roles.
active_sessions = {
    "clinical_shift": {"dr_rossi": {"attending_physician"}},
    "research_session": {"dr_rossi": {"researcher"}},
}


In [ ]:
def rbac_allows(request: Request, session_name: str) -> bool:
    # Only roles active in this session can be used.
    active_roles = active_sessions.get(session_name, {}).get(request.user, set())

    # Activated roles must also be assigned to the user.
    assigned_roles = user_roles.get(request.user, set())
    valid_active_roles = active_roles.intersection(assigned_roles)

    # Check whether any active role grants the requested operation on this resource type.
    resource_type = resources[request.resource]["type"]
    needed_permission = (request.operation, resource_type)
    return any(needed_permission in role_permissions[role] for role in valid_active_roles)


for request in requests:
    print(request.user, request.operation, request.resource, "=>", rbac_allows(request, "clinical_shift"))


In [ ]:
# Changing active roles changes the decision without changing the user's identity.
clinical_request = Request("dr_rossi", "read", "ehr_note_42", {})
print("Clinical session:", rbac_allows(clinical_request, "clinical_shift"))
print("Research session:", rbac_allows(clinical_request, "research_session"))


In [ ]:
# Dynamic separation of duty can block unsafe role combinations in one session.
conflicting_roles = [{"attending_physician", "researcher"}]


def session_is_allowed(active_roles: set[str]) -> bool:
    # A session is denied if it activates any forbidden combination.
    return not any(conflict.issubset(active_roles) for conflict in conflicting_roles)


candidate_roles = {"attending_physician", "researcher"}
print("Can activate both roles together?", session_is_allowed(candidate_roles))


## What to notice

RBAC is often the most intuitive model for hospitals because it follows job responsibilities.
The important administrative work is policy design: defining roles, avoiding role explosion,
reviewing assignments, and enforcing constraints such as separation of duty.
